# Лабораторная работа 1

**Тема:** построение простейшего поточечного графика по данным из внешнего файла.

**Выполнил:** Ипатенков Григорий Александрович, ИВТ-4, вариант 6

Для построения графика выбран Matplotlib: в задании явно указана функция `scatter()` для
точечного графика, а Matplotlib даёт прямой доступ к её параметрам (размер и цвет точек,
прозрачность, стиль маркера, сетка, легенда, оси). Pandas (`df.plot(kind='line')`) хорош для
быстрого графика из готовой таблицы, но не предназначен для ручной настройки точек. Seaborn
(`sns.lineplot`) удобен для статистической визуализации, но добавляет лишний слой абстракции
там, где нужен именно `scatter()`. Поэтому для этой задачи Matplotlib подходит точнее всего.


## Импорт библиотек

NumPy используется для вычисления шага между точками по X, Matplotlib — для построения
точечных графиков.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np


## Чтение наборов данных

Файл может содержать несколько наборов точек. Пустая строка или строка, начинающаяся с `#`,
отделяет один набор от другого. Каждая строка с данными — это пара вещественных чисел `(x, y)`.


In [ ]:
def read_datasets(filename):
    datasets = []
    current_dataset = []

    with open(filename, 'r', encoding='utf-8') as file:
        for raw_line in file:
            line = raw_line.strip()

            if not line or line.startswith('#'):
                if current_dataset:
                    datasets.append(current_dataset)
                    current_dataset = []
                continue

            parts = line.split()
            if len(parts) < 2:
                raise ValueError(f'В строке недостаточно данных: {raw_line!r}')

            x, y = float(parts[0]), float(parts[1])
            current_dataset.append((x, y))

    if current_dataset:
        datasets.append(current_dataset)

    return datasets


## Загрузка данных из внешнего файла


In [ ]:
default_file = 'lab1_data.txt'
filename = input(f'Введите имя файла с данными [{default_file}]: ').strip() or default_file

data_file = Path(filename)
datasets = read_datasets(data_file)

print(f'Файл: {data_file}')
print(f'Найдено наборов данных: {len(datasets)}')
for index, dataset in enumerate(datasets, start=1):
    print(f'Набор {index}: {len(dataset)} точек')

## Проверка неравномерного шага

По заданию нужно учитывать случай, когда точки расположены неравномерно. Поэтому график
строится по фактическим координатам из файла, без предположения о постоянном шаге.


In [ ]:
for dataset_index, dataset in enumerate(datasets, start=1):
    x_values = [point[0] for point in dataset]
    steps = np.diff(sorted(x_values))
    print(f'Набор {dataset_index}, шаги по X (после сортировки): {steps}')


## Функции построения графиков


In [ ]:
def plot_dataset(dataset, index, *, color=None, marker='o'):
    ordered = sorted(dataset, key=lambda p: p[0])
    x_values = [point[0] for point in ordered]
    y_values = [point[1] for point in ordered]

    plt.scatter(
        x_values,
        y_values,
        label=f'Набор данных #{index + 1}',
        s=55,
        alpha=0.75,
        edgecolors='black',
        linewidths=0.5,
        marker=marker,
        c=color,
    )


def format_plot(title):
    plt.title(title)
    plt.xlabel('Ось X')
    plt.ylabel('Ось Y')
    plt.grid(True, linestyle='--', alpha=0.3)
    plt.legend()
    plt.show()


## Интерактивный вывод одного или нескольких наборов


In [ ]:
choice = input('Что вывести? 1 - один набор, 2 - все наборы [2]: ').strip() or '2'

colors = ['tab:blue', 'tab:red', 'tab:green', 'tab:purple', 'tab:orange']
markers = ['o', '^', 's', 'D', '*']

if choice == '1':
    selected_number = input(f'Введите номер набора от 1 до {len(datasets)} [1]: ').strip() or '1'
    selected_index = int(selected_number) - 1
    if not 0 <= selected_index < len(datasets):
        raise ValueError('Неверный номер набора')

    plt.figure(figsize=(9, 6))
    plot_dataset(
        datasets[selected_index],
        selected_index,
        color=colors[selected_index % len(colors)],
        marker=markers[selected_index % len(markers)],
    )
    format_plot(f'Точечный график: набор данных #{selected_index + 1}')
elif choice == '2':
    plt.figure(figsize=(10, 6))
    for index, dataset in enumerate(datasets):
        plot_dataset(
            dataset,
            index,
            color=colors[index % len(colors)],
            marker=markers[index % len(markers)],
        )
    format_plot('Сравнение нескольких наборов данных')
else:
    raise ValueError('Нужно выбрать 1 или 2')

## Вывод

Программа считывает данные из внешнего файла, разделяет несколько наборов данных по пустой
строке или `#`, и строит точечный график для одного выбранного набора или сразу для нескольких
наборов на одной координатной плоскости. Неравномерность расположения точек по X не мешает
построению: перед отрисовкой набор сортируется по x, а координаты берутся напрямую из файла.
